# Slice a vault into ribs

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
from compas.datastructures import Mesh
from compas.geometry import Line, Plane, intersection_segment_plane
from compas_notebook import Viewer
from compas_notebook.config import Config

## Vault

In [ ]:
mesh = Mesh.from_meshgrid(dx=10, nx=20)
for vertex in mesh.vertices():
    x, y, _ = mesh.vertex_coordinates(vertex)
    mesh.vertex_attribute(vertex, "z", 3 * x * (10 - x) * y * (10 - y) / 625)

## Parallel planes cut each face

In [ ]:
ribs = []
for x in [1.25, 2.5, 3.75, 5.0, 6.25, 7.5, 8.75]:
    plane = Plane([x + 1e-6, 0, 0], [1, 0, 0])
    for face in mesh.faces():
        points = []
        for edge in mesh.face_halfedges(face):
            point = intersection_segment_plane(mesh.edge_line(edge), plane)
            if point:
                points.append(point)
        if len(points) == 2:
            ribs.append(Line(*points))
len(ribs)

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [5, -9, 9]
config.view.camera.target = [5, 5, 1]
viewer = Viewer(config=config)
viewer.scene.add(mesh, color="#cccccc", show_edges=False, opacity=0.5)
for rib in ribs:
    viewer.scene.add(rib, color="#dc4b8d")
viewer.show()